<a href="https://colab.research.google.com/github/poggerssLL/Automatica---Grupo-2/blob/main/etapa-02-grafos/16_Problemas_Eulerianos_e_Inspecao_de_Infraestrutura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 16 - Notebook: Circuitos Eulerianos e Inspeção da Máquina de Envase

**UNIFEI — Engenharia de Controle e Automação | Disciplina: Automática**
**Projeto: SCADA - Máquina de Envasamento de Copos Plásticos**

Neste notebook aplicamos o **Teorema de Euler** e o **Algoritmo de Hierholzer** a três problemas reais da envasadora:

1. **Ronda de inspeção preventiva** (grafo não-dirigido, todos os graus pares): o técnico percorre cada chicote elétrico/mangueira **exatamente uma vez**, saindo e voltando ao painel.
2. **Caça a vazamentos na rede pneumática** (grafo da Aula 12, com vértices de grau ímpar): resolvido pelo **Problema do Carteiro Chinês**.
3. **Limpeza CIP do circuito de água** (grafo **dirigido**: a solução de limpeza só pode escoar no sentido do fluxo).

### Melhorias em relação ao notebook original (planta NPK)
| # | Melhoria | Onde |
|---|---|---|
| M1 | Arestas **ponderadas** (tempo de inspeção) e custo total da rota | Seção 2 |
| M2 | Verificação de **conectividade** (o original só verificava paridade — condição necessária, mas não suficiente) | Seção 2 |
| M3 | Hierholzer em $O(\|E\|)$ real, com ponteiros e marcação de arestas usadas, retornando **os IDs das arestas** (o original usava `list.remove`, $O(\text{grau})$, e era ambíguo para arestas paralelas) | Seção 2 |
| M4 | **Validador** formal do circuito (cada aresta uma única vez, adjacência consecutiva, início = fim) | Seção 2 |
| M5 | Classificação **circuito / trilha / nenhum** (caso semi-euleriano com 2 vértices ímpares) | Seção 3 |
| M6 | **Carteiro Chinês** implementado (Dijkstra + emparelhamento perfeito mínimo exato por DP em bitmask) | Seções 3 e 4 |
| M7 | Versão **dirigida** ($\deg^+ = \deg^-$) com **Carteiro Chinês dirigido** para o circuito CIP | Seção 5 |
| M8 | Saída como **Ordem de Serviço** (passo a passo, com tempo acumulado) pronta para a IHM/SCADA | Seções 2, 4 e 5 |

## 1. Utilitários e Classe do Grafo de Inspeção (multigrafo não-dirigido)

In [1]:
from collections import defaultdict
from itertools import permutations
from typing import Dict, List, Tuple
import heapq

def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


class GrafoInspecaoEuleriano:
    """Multigrafo não-dirigido e ponderado: cada aresta é um trecho físico a inspecionar."""

    def __init__(self, nome: str):
        self.nome = nome
        self.arestas: List[Dict] = []
        self.adj = defaultdict(list)            # u -> [(v, indice_aresta)]

    def adicionar_trecho(self, u, v, id_trecho, tempo_min, tipo="", descricao=""):
        idx = len(self.arestas)
        self.arestas.append({"id": id_trecho, "u": u, "v": v, "tempo": tempo_min,
                             "tipo": tipo, "desc": descricao})
        self.adj[u].append((v, idx))
        self.adj[v].append((u, idx))

    def copiar(self, novo_nome):
        g = GrafoInspecaoEuleriano(novo_nome)
        for a in self.arestas:
            g.adicionar_trecho(a["u"], a["v"], a["id"], a["tempo"], a["tipo"], a["desc"])
        return g

    # ---------- propriedades estruturais ----------
    def graus(self) -> Dict[str, int]:
        return {v: len(viz) for v, viz in self.adj.items()}

    def vertices_impares(self) -> List[str]:
        return [v for v, g in self.graus().items() if g % 2 != 0]

    def conexo(self) -> bool:                                   # (M2)
        vs = [v for v, viz in self.adj.items() if viz]
        if not vs:
            return True
        vistos, pilha = {vs[0]}, [vs[0]]
        while pilha:
            u = pilha.pop()
            for w, _ in self.adj[u]:
                if w not in vistos:
                    vistos.add(w); pilha.append(w)
        return len(vistos) == len(vs)

    def classificar(self) -> Tuple[str, List[str]]:            # (M5)
        imp = self.vertices_impares()
        if not self.conexo():
            return "NENHUM (desconexo)", imp
        if len(imp) == 0:
            return "CIRCUITO EULERIANO", imp
        if len(imp) == 2:
            return "TRILHA EULERIANA", imp
        return "NENHUM (precisa do Carteiro Chinês)", imp

    def custo_total(self) -> float:
        return sum(a["tempo"] for a in self.arestas)

    # ---------- Hierholzer O(|E|) com IDs de arestas (M3) ----------
    def hierholzer(self, inicio: str) -> Tuple[List[str], List[int]]:
        usado = [False] * len(self.arestas)
        ptr = defaultdict(int)
        pilha = [(inicio, None)]
        circuito = []
        while pilha:
            u, _ = pilha[-1]
            while ptr[u] < len(self.adj[u]) and usado[self.adj[u][ptr[u]][1]]:
                ptr[u] += 1
            if ptr[u] == len(self.adj[u]):
                circuito.append(pilha.pop())
            else:
                v, idx = self.adj[u][ptr[u]]
                usado[idx] = True
                pilha.append((v, idx))
        circuito.reverse()
        return [c[0] for c in circuito], [c[1] for c in circuito[1:]]

    # ---------- validador formal (M4) ----------
    def validar_rota(self, vertices, arestas_idx, fechado=True) -> bool:
        assert sorted(arestas_idx) == list(range(len(self.arestas))), "aresta faltando ou repetida"
        for k, idx in enumerate(arestas_idx):
            a = self.arestas[idx]
            assert {vertices[k], vertices[k + 1]} == {a["u"], a["v"]}, f"passo {k+1} não é adjacente"
        if fechado:
            assert vertices[0] == vertices[-1], "rota não retorna ao início"
        return True

    # ---------- Dijkstra (Aula 14) para o Carteiro Chinês ----------
    def dijkstra(self, origem):
        dist = {v: float("inf") for v in self.adj}
        pred = {v: None for v in self.adj}                  # (vértice anterior, índice da aresta)
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        while heap:
            d, u = heapq.heappop(heap)
            if d > dist[u]:
                continue
            for v, idx in self.adj[u]:
                nd = d + self.arestas[idx]["tempo"]
                if nd < dist[v]:
                    dist[v], pred[v] = nd, (u, idx)
                    heapq.heappush(heap, (nd, v))
        return dist, pred


def emparelhamento_minimo(nos, custo):
    """Emparelhamento perfeito de custo mínimo EXATO por DP em bitmask: O(2^k * k)."""
    k = len(nos)
    memo = {}
    def dp(mask):
        if mask == (1 << k) - 1:
            return 0.0, []
        if mask in memo:
            return memo[mask]
        i = next(b for b in range(k) if not mask & (1 << b))
        melhor = (float("inf"), [])
        for j in range(i + 1, k):
            if not mask & (1 << j):
                c, pares = dp(mask | (1 << i) | (1 << j))
                c += custo[(nos[i], nos[j])]
                if c < melhor[0]:
                    melhor = (c, [(nos[i], nos[j])] + pares)
        memo[mask] = melhor
        return melhor
    return dp(0)


def carteiro_chines(g: GrafoInspecaoEuleriano):                # (M6)
    """Retorna grafo aumentado (euleriano), pares emparelhados e custo extra."""
    impares = g.vertices_impares()
    caminhos, custo = {}, {}
    for a in impares:
        dist, pred = g.dijkstra(a)
        for b in impares:
            if a != b:
                custo[(a, b)] = dist[b]
                rota, x = [], b
                while x != a:
                    x_ant, idx = pred[x]; rota.append(idx); x = x_ant
                caminhos[(a, b)] = rota[::-1]
    extra, pares = emparelhamento_minimo(impares, custo)
    g_aum = g.copiar(g.nome + " (aumentado)")
    for a, b in pares:
        for idx in caminhos[(a, b)]:
            e = g.arestas[idx]
            g_aum.adicionar_trecho(e["u"], e["v"], e["id"] + "*", e["tempo"], e["tipo"], "REPETIÇÃO")
    return g_aum, pares, extra, custo


def ordem_de_servico(g, vertices, arestas_idx, max_linhas=None):  # (M8)
    linhas, acum = [], 0.0
    for k, idx in enumerate(arestas_idx):
        a = g.arestas[idx]
        acum += a["tempo"]
        linhas.append({"Passo": k + 1, "De": vertices[k], "Para": vertices[k + 1], "Trecho": a["id"],
                       "Tipo": a["tipo"], "Tempo (min)": f"{a['tempo']:.1f}", "Acumulado": f"{acum:.1f}",
                       "Obs": a["desc"]})
    return formatar_tabela(linhas[:max_linhas] if max_linhas else linhas)

print("Classes e algoritmos carregados.")

Classes e algoritmos carregados.


## 2. Ronda de Inspeção Preventiva — Circuito Euleriano  *(M1–M4, M8)*

Os vértices são **pontos de inspeção** da máquina e as arestas são os **trechos físicos** (chicotes elétricos, mangueiras pneumáticas, eletrocalhas) que o técnico deve verificar visualmente (ressecamento, dobras, conectores frouxos). O peso de cada aresta é o tempo de inspeção do trecho.

* **Anel externo:** Painel → FRL → S100 → S200 → S300 → S400 → S500 → Painel (passa por todas as estações).
* **Anel interno:** triângulo MESA → S200 → S400 → MESA (chicotes do motor M-001 e do encoder SE-001, e cabo de rede IO-Link entre os setores 200 e 400).

Antes de executar, a pergunta de Euler é: **todos os graus são pares e o grafo é conexo?**

In [2]:
def criar_ronda_envasadora():
    g = GrafoInspecaoEuleriano("Ronda preventiva")
    # Anel externo (perímetro da máquina)
    g.adicionar_trecho("PAINEL", "FRL-001", "t1", 3.0, "Pneumático", "Linha de ar principal")
    g.adicionar_trecho("FRL-001", "S100", "t2", 2.0, "Pneumático", "Ar -> Ilha A (retentor)")
    g.adicionar_trecho("S100", "S200", "t3", 2.5, "Misto", "Sensores ZS-102/ZS-200")
    g.adicionar_trecho("S200", "S300", "t4", 3.0, "Misto", "Dosador -> pick-and-place")
    g.adicionar_trecho("S300", "S400", "t5", 2.5, "Misto", "Vácuo VAC-301 / prensa")
    g.adicionar_trecho("S400", "S500", "t6", 3.5, "Elétrico", "Resistência HT-401 / TIT-401")
    g.adicionar_trecho("S500", "PAINEL", "t7", 3.0, "Elétrico", "Motor esteira M-002")
    # Anel interno: mesa indexadora + barramento de sinais (mantém todos os graus pares)
    g.adicionar_trecho("MESA", "S200", "t8", 1.5, "Elétrico", "Chicote encoder SE-001 -> S200")
    g.adicionar_trecho("S200", "S400", "t9", 2.0, "Elétrico", "Cabo de rede IO-Link S200-S400")
    g.adicionar_trecho("S400", "MESA", "t10", 1.5, "Elétrico", "Chicote motor M-001 -> S400")
    return g

ronda = criar_ronda_envasadora()
print(formatar_tabela([{"Vértice": v, "Grau": d, "Paridade": "par" if d % 2 == 0 else "ÍMPAR"}
                       for v, d in ronda.graus().items()]))
print(f"\nLema do Aperto de Mãos: soma dos graus = {sum(ronda.graus().values())} = 2|E| = {2 * len(ronda.arestas)}")

tipo, imp = ronda.classificar()
print(f"Conexo: {ronda.conexo()} | Vértices ímpares: {imp} | Classificação: {tipo}")
assert tipo == "CIRCUITO EULERIANO"

rota_v, rota_e = ronda.hierholzer("PAINEL")
ronda.validar_rota(rota_v, rota_e)
print("\nCircuito Euleriano:", " -> ".join(rota_v))
print("✓ Validado: cada um dos", len(ronda.arestas), "trechos inspecionado exatamente 1 vez, retorno ao PAINEL.")
print(f"Tempo total da ronda = soma dos pesos = {ronda.custo_total():.1f} min (ótimo: nenhuma repetição)\n")
print("=== ORDEM DE SERVIÇO — RONDA PREVENTIVA ===")
print(ordem_de_servico(ronda, rota_v, rota_e))

Vértice | Grau | Paridade
--------+------+---------
PAINEL  | 2    | par     
FRL-001 | 2    | par     
S100    | 2    | par     
S200    | 4    | par     
S300    | 2    | par     
S400    | 4    | par     
S500    | 2    | par     
MESA    | 2    | par     

Lema do Aperto de Mãos: soma dos graus = 20 = 2|E| = 20
Conexo: True | Vértices ímpares: [] | Classificação: CIRCUITO EULERIANO

Circuito Euleriano: PAINEL -> FRL-001 -> S100 -> S200 -> S300 -> S400 -> S200 -> MESA -> S400 -> S500 -> PAINEL
✓ Validado: cada um dos 10 trechos inspecionado exatamente 1 vez, retorno ao PAINEL.
Tempo total da ronda = soma dos pesos = 24.5 min (ótimo: nenhuma repetição)

=== ORDEM DE SERVIÇO — RONDA PREVENTIVA ===
Passo | De      | Para    | Trecho | Tipo       | Tempo (min) | Acumulado | Obs                           
------+---------+---------+--------+------------+-------------+-----------+-------------------------------
1     | PAINEL  | FRL-001 | t1     | Pneumático | 3.0         | 3.0       | Li

## 3. Quebrando a Paridade: Trilha Euleriana e Carteiro Chinês  *(M5, M6)*

Suponha que a equipe instale uma **mangueira de by-pass** de água entre S200 (envase) e S300 (tampa), que também precisa ser inspecionada. Os graus de S200 e S300 passam a ser ímpares:

* Com **2 vértices ímpares** existe uma **trilha** euleriana (aberta), que começa em um ímpar e termina no outro — útil se o técnico puder terminar a ronda longe do painel.
* Se a ronda **precisa** voltar ao painel, aplicamos o **Carteiro Chinês**: repetimos o caminho mais barato entre os ímpares.

In [3]:
ronda2 = criar_ronda_envasadora()
ronda2.adicionar_trecho("S200", "S300", "t11", 2.0, "Hidráulico", "By-pass de água (novo)")
tipo2, imp2 = ronda2.classificar()
print(f"Vértices ímpares: {imp2} | Classificação: {tipo2}")

# (a) Trilha euleriana: começa em um vértice ímpar
tv, te = ronda2.hierholzer(imp2[0])
ronda2.validar_rota(tv, te, fechado=False)
print(f"\n(a) Trilha Euleriana ({tv[0]} ... {tv[-1]}):", " -> ".join(tv))
print(f"    Custo = {ronda2.custo_total():.1f} min, mas termina em {tv[-1]}, não no PAINEL.")

# (b) Carteiro Chinês: circuito fechado a partir do PAINEL
g_aum, pares, extra, _ = carteiro_chines(ronda2)
cv, ce = g_aum.hierholzer("PAINEL")
g_aum.validar_rota(cv, ce)
print(f"\n(b) Carteiro Chinês: emparelhamento {pares}, custo extra = {extra:.1f} min")
print("    Trechos repetidos:", [a["id"] for a in g_aum.arestas if a["desc"] == "REPETIÇÃO"])
print("    Circuito:", " -> ".join(cv))
print(f"    Custo total = {ronda2.custo_total():.1f} + {extra:.1f} = {g_aum.custo_total():.1f} min")

Vértices ímpares: ['S200', 'S300'] | Classificação: TRILHA EULERIANA

(a) Trilha Euleriana (S200 ... S300): S200 -> S100 -> FRL-001 -> PAINEL -> S500 -> S400 -> S300 -> S200 -> MESA -> S400 -> S200 -> S300
    Custo = 26.5 min, mas termina em S300, não no PAINEL.

(b) Carteiro Chinês: emparelhamento [('S200', 'S300')], custo extra = 2.0 min
    Trechos repetidos: ['t11*']
    Circuito: PAINEL -> FRL-001 -> S100 -> S200 -> S300 -> S400 -> S200 -> S300 -> S200 -> MESA -> S400 -> S500 -> PAINEL
    Custo total = 26.5 + 2.0 = 28.5 min


## 4. Caça a Vazamentos na Rede Pneumática — Carteiro Chinês Completo  *(M6, M8)*

Na Aula 12 o resíduo do balanço $B\vec{Q} - \vec{S}$ acusou um vazamento na `ILHA-B`. Agora o técnico precisa **varrer todas as mangueiras com detector ultrassônico**, saindo e voltando ao `FRL-001`.

A rede pneumática é praticamente uma **árvore**: cada cilindro é uma **folha (grau 1)**, logo há muitos vértices ímpares e nenhum circuito euleriano existe. Peso = comprimento (m) × 2 min/m (velocidade de varredura do detector).

In [4]:
MIN_POR_M = 2.0
trechos_pneu = [  # (origem, destino, id, comprimento_m) — mesma rede da Aula 12
    ("FRL-001", "ILHA-A", "LINHA-01", 2.0), ("FRL-001", "ILHA-B", "LINHA-02", 4.0),
    ("ILHA-A", "ILHA-B", "LINHA-EQ", 1.5),
    ("ILHA-A", "CIL-A", "XV-101", 0.8), ("ILHA-A", "CIL-B", "XV-201", 1.0),
    ("ILHA-A", "CIL-C", "XV-202", 1.2), ("ILHA-A", "BICO", "XV-203", 1.0),
    ("ILHA-B", "CIL-D", "XV-301", 1.0), ("ILHA-B", "CIL-E", "XV-302", 1.2),
    ("ILHA-B", "EJT-301", "VAC-301", 1.2), ("ILHA-B", "CIL-F", "XV-401", 0.8),
    ("ILHA-B", "CIL-G", "XV-501", 1.0), ("ILHA-B", "CIL-H", "XV-502", 1.5),
]
rede = GrafoInspecaoEuleriano("Rede pneumática")
for u, v, i, L in trechos_pneu:
    rede.adicionar_trecho(u, v, i, L * MIN_POR_M, "Pneumático", f"{L} m")

tipo3, imp3 = rede.classificar()
print(f"Graus: {rede.graus()}")
print(f"Vértices ímpares ({len(imp3)}): {imp3}")
print(f"Classificação: {tipo3}")

g_pneu, pares3, extra3, custo3 = carteiro_chines(rede)
cv3, ce3 = g_pneu.hierholzer("FRL-001")
g_pneu.validar_rota(cv3, ce3)

print(f"\nEmparelhamento perfeito mínimo ({len(pares3)} pares, entre "
      f"{len(imp3) * (len(imp3) - 1) // 2} distâncias calculadas):")
print(formatar_tabela([{"Par": f"{a} <-> {b}", "Custo (min)": f"{custo3[(a, b)]:.1f}"} for a, b in pares3]))
print(f"\nTempo sem repetição (limite inferior): {rede.custo_total():.1f} min")
print(f"Tempo extra (repetições):              {extra3:.1f} min")
print(f"Tempo total ótimo da varredura:        {g_pneu.custo_total():.1f} min "
      f"(+{100 * extra3 / rede.custo_total():.0f} %)")
print("\n=== ORDEM DE SERVIÇO — CAÇA A VAZAMENTOS (FRL-001 -> ... -> FRL-001) ===")
print(ordem_de_servico(g_pneu, cv3, ce3))

Graus: {'FRL-001': 2, 'ILHA-A': 6, 'ILHA-B': 8, 'CIL-A': 1, 'CIL-B': 1, 'CIL-C': 1, 'BICO': 1, 'CIL-D': 1, 'CIL-E': 1, 'EJT-301': 1, 'CIL-F': 1, 'CIL-G': 1, 'CIL-H': 1}
Vértices ímpares (10): ['CIL-A', 'CIL-B', 'CIL-C', 'BICO', 'CIL-D', 'CIL-E', 'EJT-301', 'CIL-F', 'CIL-G', 'CIL-H']
Classificação: NENHUM (precisa do Carteiro Chinês)

Emparelhamento perfeito mínimo (5 pares, entre 45 distâncias calculadas):
Par               | Custo (min)
------------------+------------
CIL-A <-> CIL-B   | 3.6        
CIL-C <-> BICO    | 4.4        
CIL-D <-> CIL-G   | 4.0        
CIL-E <-> EJT-301 | 4.8        
CIL-F <-> CIL-H   | 4.6        

Tempo sem repetição (limite inferior): 36.4 min
Tempo extra (repetições):              21.4 min
Tempo total ótimo da varredura:        57.8 min (+59 %)

=== ORDEM DE SERVIÇO — CAÇA A VAZAMENTOS (FRL-001 -> ... -> FRL-001) ===
Passo | De      | Para    | Trecho   | Tipo       | Tempo (min) | Acumulado | Obs      
------+---------+---------+----------+------------+

## 5. Limpeza CIP do Circuito de Água — Euler em Dígrafo  *(M7, M8)*

Na sanitização *Clean-In-Place*, a solução de limpeza sai do tanque CIP, passa por **todos os trechos do circuito de água no sentido do fluxo** e retorna ao tanque. Cada trecho deve ser lavado (pelo menos) uma vez, e **não pode ser percorrido ao contrário** → grafo **dirigido**.

**Critério:** existe circuito euleriano dirigido $\iff$ o grafo é (fracamente) conexo e $\deg^+(v) = \deg^-(v)$ para todo $v$.

* A válvula de 3 vias `V3V-201` recebe água do reservatório, envia ao dosador, recebe de volta e envia ao bico (grau de entrada = 2 e grau de saída = 2).
* O projeto inicial possui um **dreno de fundo** `TQ-201 → BANDEJA`, que desbalanceia o grafo.

In [5]:
class GrafoCIP:
    """Multigrafo dirigido ponderado (tempo de lavagem do trecho, em min)."""
    def __init__(self):
        self.arestas, self.adj = [], defaultdict(list)

    def adicionar(self, u, v, id_e, tempo, desc=""):
        self.arestas.append({"id": id_e, "u": u, "v": v, "tempo": tempo, "tipo": "CIP", "desc": desc})
        self.adj[u].append((v, len(self.arestas) - 1))
        self.adj[v]                                           # garante o vértice no dicionário

    def saldos(self):                                          # deg+ - deg-
        s = defaultdict(int)
        for a in self.arestas:
            s[a["u"]] += 1; s[a["v"]] -= 1
        return {v: s[v] for v in self.adj}

    def hierholzer(self, inicio):
        ptr, pilha, circ = defaultdict(int), [(inicio, None)], []
        while pilha:
            u, _ = pilha[-1]
            if ptr[u] < len(self.adj[u]):
                v, idx = self.adj[u][ptr[u]]; ptr[u] += 1
                pilha.append((v, idx))
            else:
                circ.append(pilha.pop())
        circ.reverse()
        return [c[0] for c in circ], [c[1] for c in circ[1:]]

    def dijkstra(self, origem):
        dist, pred = {v: float("inf") for v in self.adj}, {v: None for v in self.adj}
        dist[origem], heap = 0.0, [(0.0, origem)]
        while heap:
            d, u = heapq.heappop(heap)
            if d > dist[u]:
                continue
            for v, idx in self.adj[u]:
                if d + self.arestas[idx]["tempo"] < dist[v]:
                    dist[v], pred[v] = d + self.arestas[idx]["tempo"], (u, idx)
                    heapq.heappush(heap, (dist[v], v))
        return dist, pred

    def balancear_carteiro_dirigido(self):
        """Carteiro Chinês dirigido: liga vértices com excesso de entrada aos com excesso de saída."""
        s = self.saldos()
        sobra_entrada = [v for v, x in s.items() if x < 0 for _ in range(-x)]   # precisam de saída extra
        sobra_saida = [v for v, x in s.items() if x > 0 for _ in range(x)]      # precisam de entrada extra
        melhor = (float("inf"), None)
        for perm in permutations(sobra_saida):                  # atribuição exata (poucos nós)
            c = sum(self.dijkstra(a)[0][b] for a, b in zip(sobra_entrada, perm))
            if c < melhor[0]:
                melhor = (c, list(zip(sobra_entrada, perm)))
        for a, b in melhor[1]:
            _, pred = self.dijkstra(a)
            x, rota = b, []
            while x != a:
                x_ant, idx = pred[x]; rota.append(idx); x = x_ant
            for idx in reversed(rota):
                e = self.arestas[idx]
                self.adicionar(e["u"], e["v"], e["id"] + "*", e["tempo"], "REPETIÇÃO")
        return melhor

cip = GrafoCIP()
cip.adicionar("TQ-CIP", "B-CIP", "c1", 1.0, "Sucção da bomba CIP")
cip.adicionar("B-CIP", "TQ-201", "c2", 3.0, "Spray-ball do reservatório")
cip.adicionar("TQ-201", "V3V-201", "c3", 1.5, "Sucção do reservatório")
cip.adicionar("V3V-201", "DOS-202", "c4", 2.0, "Enchimento da câmara do dosador")
cip.adicionar("DOS-202", "V3V-201", "c5", 2.0, "Expulsão da dose")
cip.adicionar("V3V-201", "BICO-203", "c6", 1.5, "Linha do bico")
cip.adicionar("BICO-203", "BANDEJA", "c7", 1.0, "Bico -> copo coletor")
cip.adicionar("BANDEJA", "TQ-CIP", "c8", 1.5, "Retorno ao tanque CIP")
cip.adicionar("TQ-201", "BANDEJA", "c9", 1.0, "Dreno de fundo do reservatório")

print(formatar_tabela([{"Vértice": v, "deg+": sum(1 for a in cip.arestas if a["u"] == v),
                        "deg-": sum(1 for a in cip.arestas if a["v"] == v),
                        "Saldo": f"{x:+d}", "Status": "OK" if x == 0 else "DESBALANCEADO"}
                       for v, x in cip.saldos().items()]))
desbal = {v: x for v, x in cip.saldos().items() if x}
print(f"\nCircuito euleriano dirigido? {not desbal} -> vértices desbalanceados: {desbal}")

custo_extra, pares_cip = cip.balancear_carteiro_dirigido()
print(f"\nCarteiro Chinês dirigido: ligar {pares_cip} -> custo extra = {custo_extra:.1f} min")
print("Trechos lavados 2 vezes:", [a["id"] for a in cip.arestas if a["desc"] == "REPETIÇÃO"])
assert all(x == 0 for x in cip.saldos().values())

cv4, ce4 = cip.hierholzer("TQ-CIP")
assert sorted(ce4) == list(range(len(cip.arestas))) and cv4[0] == cv4[-1] == "TQ-CIP"
print("\nCircuito CIP:", " -> ".join(cv4))
print(f"Tempo total de CIP: {sum(a['tempo'] for a in cip.arestas):.1f} min\n")
print("=== ORDEM DE SERVIÇO — SEQUÊNCIA DE LAVAGEM CIP ===")
print(ordem_de_servico(cip, cv4, ce4))

Vértice  | deg+ | deg- | Saldo | Status       
---------+------+------+-------+--------------
TQ-CIP   | 1    | 1    | +0    | OK           
B-CIP    | 1    | 1    | +0    | OK           
TQ-201   | 2    | 1    | +1    | DESBALANCEADO
V3V-201  | 2    | 2    | +0    | OK           
DOS-202  | 1    | 1    | +0    | OK           
BICO-203 | 1    | 1    | +0    | OK           
BANDEJA  | 1    | 2    | -1    | DESBALANCEADO

Circuito euleriano dirigido? False -> vértices desbalanceados: {'TQ-201': 1, 'BANDEJA': -1}

Carteiro Chinês dirigido: ligar [('BANDEJA', 'TQ-201')] -> custo extra = 5.5 min
Trechos lavados 2 vezes: ['c8*', 'c1*', 'c2*']

Circuito CIP: TQ-CIP -> B-CIP -> TQ-201 -> V3V-201 -> DOS-202 -> V3V-201 -> BICO-203 -> BANDEJA -> TQ-CIP -> B-CIP -> TQ-201 -> BANDEJA -> TQ-CIP
Tempo total de CIP: 20.0 min

=== ORDEM DE SERVIÇO — SEQUÊNCIA DE LAVAGEM CIP ===
Passo | De       | Para     | Trecho | Tipo | Tempo (min) | Acumulado | Obs                            
------+----------+----

## 6. Conclusões

* **Paridade + conectividade** decidem, em $O(|V| + |E|)$, se existe rota sem repetição; o Hierholzer a constrói em $O(|E|)$.
* A ronda preventiva foi **projetada** com graus pares → custo mínimo = soma dos pesos (nenhum desperdício).
* A rede pneumática real é quase uma árvore: **toda mangueira terminal precisa ser percorrida duas vezes**. O Carteiro Chinês quantifica esse custo e mostra que concentrar consumidores em menos ilhas (ou instalar a ilha junto ao atuador) reduz o tempo de manutenção.
* No CIP, o desbalanceamento causado pelo dreno é detectado automaticamente e corrigido com o **mínimo de relavagem**, o que reduz o consumo de água e de químicos.